# Lesson 1.2 — Coordinator / Subagent Orchestration (Hub-and-Spoke)

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Describe hub-and-spoke architecture where a coordinator routes all inter-subagent communication
- Explain why subagents operate with isolated context and do not inherit coordinator history
- Dynamically select which subagents to invoke based on query requirements
- Add an iterative refinement loop where the coordinator re-delegates to close coverage gaps

**You will produce:** A coordinator that decomposes a topic, delegates to subagents, and refines on coverage gaps.

## Concepts

In **hub-and-spoke** orchestration a single **coordinator** owns decomposition, delegation, result aggregation, error handling, and information routing. Subagents are the spokes; they never talk to each other directly.

- **Isolated context:** a subagent does NOT inherit the coordinator's conversation history. Everything it needs must be placed in its prompt (Lesson 1.3 covers the mechanics).
- **Dynamic selection:** a good coordinator analyzes the query and invokes only the subagents needed, rather than always running the full pipeline.
- **Observability:** routing all communication through the coordinator gives one place for consistent error handling and controlled information flow.

### Exam anti-pattern: overly narrow decomposition
Sample Question 7 shows a system that decomposed "impact of AI on **creative industries**" into only *digital art, graphic design, photography* - missing music, writing, and film. Each subagent worked correctly; the **coordinator's decomposition was too narrow**. Mitigate with an iterative refinement loop that evaluates synthesis output for coverage gaps and re-delegates targeted queries.

> **Reference:** [Anthropic - Agent SDK (subagents)](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Model the hub-and-spoke roles

We represent subagents as role specs. Only the coordinator holds the shared plan.

In [ ]:
SUBAGENTS = {
    "web_search": "Finds and returns relevant web sources for a subtopic.",
    "doc_analysis": "Summarizes documents/papers for a subtopic.",
    "synthesis": "Combines findings into a cited report; flags coverage gaps.",
}

def coordinator_prompt(topic):
    # Specify GOALS and quality criteria, not step-by-step procedure.
    return (
        f"Research topic: {topic}.\n"
        "Decompose into subtopics that COMPLETELY cover the topic's major domains. "
        "List subtopics as a JSON array. Prefer breadth over depth at this stage."
    )

print(coordinator_prompt("impact of AI on creative industries"))

## Code-Along: Step 2 — Decompose, then check for coverage gaps

The coordinator decomposes, then a gap check re-delegates missing domains. This is the defense against narrow decomposition from Sample Question 7.

In [ ]:
EXPECTED_DOMAINS = {"visual arts", "music", "writing", "film"}

def decompose(topic):
    if HAVE_API:
        resp = client.messages.create(model=MODEL_SONNET, max_tokens=512,
            messages=[{"role": "user", "content": coordinator_prompt(topic)}])
        return "".join(b.text for b in resp.content if b.type == "text")
    return '["AI in digital art", "AI in graphic design", "AI in photography"]'

def coverage_gaps(subtopics_text):
    t = subtopics_text.lower()
    covered = {d for d in EXPECTED_DOMAINS if d in t or d.split()[0] in t}
    return EXPECTED_DOMAINS - covered

subs = decompose("impact of AI on creative industries")
print("subtopics:", subs)
print("coverage gaps:", coverage_gaps(subs))

## Code-Along: Step 3 — Iterative refinement loop

If gaps remain, the coordinator re-delegates targeted subtasks until coverage is sufficient.

In [ ]:
def refine(topic, max_rounds=3):
    subs = decompose(topic)
    for _ in range(max_rounds):
        gaps = coverage_gaps(subs)
        if not gaps:
            break
        # Re-delegate: append explicit missing domains (shown here as augmentation)
        subs = subs + " + " + ", ".join(f"AI in {g}" for g in gaps)
    return subs, coverage_gaps(subs)

final, remaining = refine("impact of AI on creative industries")
print("final subtopics:", final)
print("remaining gaps:", remaining)

## Exercise

Write a `select_subagents(query)` function that returns only the subagents a query needs (e.g. a pure fact lookup should NOT invoke `doc_analysis`). Then prove that your `refine` loop closes a seeded gap by starting from a deliberately narrow decomposition.

In [ ]:
# TODO: implement select_subagents(query) -> list of subagent keys
# TODO: show a query that invokes only {'web_search'}
# TODO: start from ['AI in photography'] and assert refine() reaches full coverage

## Key Takeaways

- The coordinator owns decomposition, delegation, aggregation, routing, and error handling.
- Subagents have isolated context - they never inherit coordinator history automatically.
- Dynamically select subagents; don't always run the full pipeline.
- Overly narrow decomposition causes coverage gaps - add an iterative refinement loop.

---

**Next:** [Lesson 1.3 — Subagent Invocation & Context Passing](./lesson-1.3-subagent-context-passing.ipynb)